# Importação das bibliotecas necessárias

In [8]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [9]:
# Crie uma instância da classe
bucket = "silver"
datamart = "SGT"
extrator_bronze = AlinharETL(bucket,datamart)

2024-11-19 15:23:43,370 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [10]:
df_relatorios = extrator_bronze.criar_view_temporaria('bronze/sgt/relatorios','relatorios_sgt')

2024-11-19 15:23:44,240 - INFO - Aguarde... Criando View (bronze/sgt/relatorios)
2024-11-19 15:23:44,860 - INFO - View criada com sucesso


In [12]:
df_relatorios.show()

+-----------------+----+--------------+--------------+-------------------------+---------------------+------------------------+------------------------+---+--------------------+-----------------+-------+-------------+--------------+---------------+--------+---+--------------------+--------------------+-----------------+-------------+--------------------+--------------------+--------------------+
|CRProdutoRegional| ano|cpfCnpjCliente|cpfColaborador|descricaoProdutoCategoria|descricaoProdutoLinha|descricaoProdutoNacional|descricaoProdutoRegional|dia|    emailColaborador|ensaiosRealizados|     id|idAtendimento|idPorteCliente|isCompartilhado|isEmRede|mes|         nomeCliente|     nomeColaborador|numeroAtendimento| porteCliente|relatoriosRealizados|   tituloAtendimento|  unidadeOperacional|
+-----------------+----+--------------+--------------+-------------------------+---------------------+------------------------+------------------------+---+--------------------+-----------------+-------

In [4]:
sql_query = """
SELECT 
    distinct
    TO_DATE(CONCAT_WS('-', ano, mes, dia)) AS id_data_calendario,
    CRProdutoRegional AS cr_produto_regional,
    ano AS ano,
    cpfCnpjCliente AS nr_cnpj_cliente,
    cpfColaborador AS nr_cpf_colaborador,
    descricaoProdutoCategoria AS dsc_produto_categoria,
    descricaoProdutoLinha AS dsc_produto_linha,
    descricaoProdutoNacional AS dsc_produto_nacional,
    descricaoProdutoRegional AS dsc_produto_regional,
    dia AS dia,
    emailColaborador AS nm_email_colaborador,
    ensaiosRealizados AS nr_ensaios_realizados,
    id AS id,
    idAtendimento AS id_atendimento,
    CASE 
        WHEN idPorteCliente LIKE '%@nil%' THEN regexp_extract(idPorteCliente, '@nil":"([^"]+)', 1)
        ELSE idPorteCliente 
    END AS id_porte_cliente,
    isCompartilhado AS dsc_compartilhamento,
    isEmRede AS dsc_em_rede,
    mes AS mes,
    nomeCliente AS nm_cliente,
    CASE 
        WHEN nomeColaborador LIKE '%@nil%' THEN regexp_extract(nomeColaborador, '@nil":"([^"]+)', 1)
        ELSE nomeColaborador 
    END AS nm_colaborador,
    CASE 
        WHEN numeroAtendimento LIKE '%@nil%' THEN regexp_extract(numeroAtendimento, '@nil":"([^"]+)', 1)
        ELSE numeroAtendimento 
    END AS nr_atendimento,
    CASE 
        WHEN porteCliente LIKE '%@nil%' THEN regexp_extract(porteCliente, '@nil":"([^"]+)', 1)
        ELSE porteCliente 
    END AS nm_porte_cliente,
    relatoriosRealizados AS nr_relatorios_realizados,
    tituloAtendimento AS dsc_titulo_atendimento,
    unidadeOperacional AS nm_unidade_operacional
FROM 
    relatorios_sgt
"""

In [5]:
df_relatorios = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-26 16:33:47,372 - INFO - Aguarde... Executando Query (Delta)
2024-09-26 16:33:47,373 - INFO - 
SELECT 
    
    TO_DATE(CONCAT_WS('-', ano, mes, dia)) AS id_data_calendario,
    CRProdutoRegional AS cr_produto_regional,
    ano AS ano,
    cpfCnpjCliente AS nr_cnpj_cliente,
    cpfColaborador AS nr_cpf_colaborador,
    descricaoProdutoCategoria AS dsc_produto_categoria,
    descricaoProdutoLinha AS dsc_produto_linha,
    descricaoProdutoNacional AS dsc_produto_nacional,
    descricaoProdutoRegional AS dsc_produto_regional,
    dia AS dia,
    emailColaborador AS nm_email_colaborador,
    ensaiosRealizados AS nr_ensaios_realizados,
    id AS id,
    idAtendimento AS id_atendimento,
    CASE 
        WHEN idPorteCliente LIKE '%@nil%' THEN regexp_extract(idPorteCliente, '@nil":"([^"]+)', 1)
        ELSE idPorteCliente 
    END AS id_porte_cliente,
    isCompartilhado AS dsc_compartilhamento,
    isEmRede AS dsc_em_rede,
    mes AS mes,
    nomeCliente AS nm_cliente,
    CASE 
    

# Gravação no datalake

In [6]:
extrator_bronze.caminho_tabela_delta = 's3a://silver/SGT/RelatoriosSgt'

In [7]:
extrator_bronze.salvar_delta(df_relatorios, 'overwrite')

2024-09-25 17:07:37,588 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:07:46,189 - INFO - Dados persistidos com sucesso
2024-09-25 17:07:46,190 - INFO - s3a://silver/SGT/RelatoriosSgt
2024-09-25 17:07:46,190 - INFO - Aguarde... Realizando optimize
2024-09-25 17:07:48,409 - INFO - Optimize executado com sucesso.
2024-09-25 17:07:48,410 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:08:16,035 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [13]:
extrator_bronze.parar_sessao()

2024-11-19 15:24:05,321 - INFO - Sessão Spark finalizada.
